<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [ ]:
# @title Dependencies

# Installation
!pip install pandas pyarrow -q
!pip install numpy -q
!pip install tqdm -q
!pip install openai -q
!pip install groq -q

# First-party installations
import itertools
import re
import json
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os

# Third-party installations
from google.colab import drive
import google.colab.userdata
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from openai import APIError
from groq import Groq
from google import genai
from google.genai import types

In [ ]:
# @title Paths and definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/iclr/final.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.


In [ ]:
# @title Hardcoded definitions

# Target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Models using the OpenAI-API
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-11-20", # Changed to a newer model version
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Models using the Groq-API
GROQ_MODELS = [
    "llama-3.3-70b-versatile"
]

# Models using the Gemini-API
GEMINI_MODELS = [
    "gemini-3-flash-preview" # new
]

# Models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07",
    "gemini-3-flash-preview"
}

# Define temperature parameters (as used by Cummins 2025)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins 2025)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "Explain your thought process step by step."
]

# Define note taking
NOTE_TAKING = [
    "Yes",
    "No"
]

# Define the number of reviews to generate per paper for each combo
NUM_REVIEWS_PER_PAPER = 1

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    note_taking: Literal[*NOTE_TAKING]

# Define the full result path
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_results.parquet")

# Define the full log path
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_progress.parquet")

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

In [ ]:
# Settings
MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random
SEED_VALUE = 45 # Random
MAX_TOKENS = 2000 # As Robertson and Kayejo (2025) (in their repo)

# Prompts
NOTE_TAKING_FAITHFUL = f"""

Take detailed, accurate notes on the paper for an ICLR style review.

Focus on precisely capturing the actual methods, results, and contributions without any distortion.

Just output the notes.

""" # Adopted from Robertson and Koyejo (2025)

REVIEW_GENERATION = f"""

Create an ICLR-style review following this specific structure:

# Summary Of The Paper
Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses
Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility
Evaluate based on your notes.

# Summary Of The Review
Provide a 2-3 sentence distillation of your overall assessment.

# Correctness
Rate on a scale of 1-5.

# Technical Novelty And Significance
Rate on a scale of 1-5.

# Empirical Novelty And Significance
Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

""" # Adopted from Robertson and Koyejo (2025)

# Predefined placeholder strings for consistent error/status reporting
NOTE_TAKING_NOT_ATTEMPTED = "NOTE TAKING NOT ATTEMPTED"
NOTE_TAKING_FAILURE = "ERROR: NOTE TAKING NOT SUCCESSFUL"
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE = "ERROR: UNKNOWN STATE REACHED"

# OpenAI API-key
# OPENAI_KEY = google.colab.userdata.get('OPENAI_API_KEY')

# Groq API-key
# GROQ_KEY = google.colab.userdata.get('GROQ_API_KEY')

# Gemini API-key
GEMINI_API_KEY = google.colab.userdata.get('GEMINI_API_KEY')

# Groq URL
GROQ_URL ="https://api.groq.com/openai/v1"

In [ ]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

# Select target columns
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']]

# Remove the HEADER_TO_REMOVE string from the 'parsed_text' column
paper_content['parsed_text'] = paper_content['parsed_text'].str.replace(HEADER_TO_REMOVE, '', regex=False).str.strip()

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

In [ ]:
def _schema_to_openai(chain_of_thought: str) -> Dict[str, Any]:
    """
    Creates a response schema for API calls with OpenAIs non-reasoning-models.
    Additionally it adds the chain of thought instruction to the prompt.
    """

    return {
        "type": "function",
        "function": {
            "name": "response_format",
            "description": "The function used to return the single, structured text response.",
            "parameters": {
                "type": "object",
                "properties": {
                    "final_response": {
                        "type": "string",
                        "description": (
                            f"Generate the complete, continuous peer-review based on the provided content. {chain_of_thought}"
                        )
                    }
                },
                "required": ["final_response"]
            }
        }
    }

def _extract_json(text: str) -> dict:
    """Extract and parse a JSON object from a potentially messy API response"""
    if not text:
        return {}

    # Strip code fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Find first {...}
    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return {}

### API

In [ ]:
class LLMClient:
    def __init__(self, seed: int, simulate: bool = True):
        """Initialize LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)

        # Placeholders for real clients when simulate=False:
        self._openai = None
        self._groq = None
        self._gemini_client = None

    def _maybe_init_clients(self):
        """Start simulation or API clients"""
        # Simulation
        if self.simulate:
            return
        # Initialize OpenAI client
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)

        # Initialize Groq client
        if self._groq is None and "GROQ_KEY" in globals():
            self._groq = Groq(api_key=GROQ_KEY, base_url=GROQ_URL)

        # Configure Gemini API globally and instantiate client
        if self._gemini_client is None and "GEMINI_API_KEY" in globals():
            self._gemini_client = genai.Client(api_key=GEMINI_API_KEY)

    def call(
        self,
        model: str,
        initial_prompt_content: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        chain_of_thought: str,
        note_taking_active: str,
        max_retries: float = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        max_tokens: int = MAX_TOKENS,
    ) -> Tuple[str, str, str]:
        """
        Defines the API calls / simulations. If `note_taking_active` is True,
        notes are generated first and then used as input for the main review generation.
        Otherwise, `initial_prompt_content` is directly used for review generation.
        Returns: (notes_content_for_dataframe, raw_review_output, parsed_review_output)
        """

        self._maybe_init_clients()

        # Initialize review_input_content; it will be overwritten by generated notes if note_taking is active and successful.
        review_input_content = initial_prompt_content
        # Initialize notes_content_for_df to reflect that note taking was not attempted by default.
        notes_content_for_df = NOTE_TAKING_NOT_ATTEMPTED

        # --- Step 1: Generate notes (only if note_taking_active is True) ---
        if note_taking_active == "Yes":
            generated_notes = None # Temporary variable to hold notes during generation attempt

            note_taking_prompt_full = NOTE_TAKING_FAITHFUL + initial_prompt_content + NOTE_TAKING_FAITHFUL

            # Add optional chain of thought instruction
            if chain_of_thought:
                note_taking_prompt_full += f" {chain_of_thought}"

            # Determine provider for note-taking model
            provider_notes = "openai"
            if model in GROQ_MODELS:
                provider_notes = "groq"
            elif model in GEMINI_MODELS:
                provider_notes = "gemini"

            for attempt in range(1, max_retries + 1):
                try:
                    # Simulate artificial note
                    if self.simulate:
                        generated_notes = (
                            f"This is a simulated note based on: Model='{model}', Temp={temperature}, "
                            f"Effort='{reasoning_effort}', CoT='{chain_of_thought}', Notes={note_taking_active}. "
                        )
                        break
                    # Call API
                    else:
                        # For OpenAI
                        if provider_notes == "openai":
                            kwargs = {}
                            if model in REASONING_MODELS and reasoning_effort:
                                kwargs["reasoning"] = {"effort": reasoning_effort}
                            if temperature is not None:
                                kwargs["temperature"] = float(temperature)

                            system_msg = {
                                "role": "system",
                                "content": "You are a helpful assistant that takes concise, accurate notes based on the provided text. Your output should be plain text and summarize the key methods, results, and contributions. Do not include any additional commentary or formatting beyond the notes themselves."
                            }
                            msgs = [system_msg, {"role": "user", "content": note_taking_prompt_full}]
                            resp = self._openai.chat.completions.create(
                                model=model,
                                messages=msgs,
                                max_tokens=max_tokens,
                                **kwargs,
                            )
                            generated_notes = resp.choices[0].message.content or ""
                            break
                        # For Groq
                        elif provider_notes == "groq":
                            gkwargs = {}
                            if temperature is not None:
                                gkwargs["temperature"] = float(temperature)
                            resp = self._groq.chat.completions.create(
                                model=model,
                                messages=[{"role": "user", "content": note_taking_prompt_full}],
                                max_tokens=max_tokens,
                                **gkwargs,
                            )
                            generated_notes = resp.choices[0].message.content.strip()
                            break
                        # For Gemini
                        elif provider_notes == "gemini":
                            gen_content_config_args = {
                                "response_mime_type": "text/plain"
                            }
                            if model in REASONING_MODELS and reasoning_effort:
                                gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                            resp = self._gemini_client.models.generate_content(
                                model=model,
                                contents=note_taking_prompt_full,
                                config=types.GenerateContentConfig(**gen_content_config_args)
                            )
                            generated_notes = resp.text
                            break

                # Raise error flag in case of failure
                except Exception as e:
                    print(f"[LLM ERROR - Note Taking] provider={{provider_notes}} model={{model}} attempt={{attempt}} -> {{e}}", flush=True)
                    if attempt == max_retries:
                        # Case: Note taking active but fails. Return early with placeholder for all three.
                        return NOTE_TAKING_FAILURE, NOTE_TAKING_FAILURE, NOTE_TAKING_FAILURE
                    time.sleep(retry_delay)

            # If the loop finishes, generated_notes should be set if successful.
            if generated_notes is not None:
                review_input_content = generated_notes
                notes_content_for_df = generated_notes
            else:
                # This path handles cases where note_taking_active was True, but generated_notes is still None after retries (e.g., unexpected internal error).
                return NOTE_TAKING_FAILURE, NOTE_TAKING_FAILURE, NOTE_TAKING_FAILURE

        # --- Step 2: Generate the main review (always executes, either with original content or generated notes) ---

        # Generate final prompt (as Robertson and Kayejo (2025))
        final_review_prompt = REVIEW_GENERATION + review_input_content + REVIEW_GENERATION

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_review_prompt += f" {chain_of_thought}"

        # Determine provider for review model
        provider_review = "openai"
        if model in GROQ_MODELS:
            provider_review = "groq"
        elif model in GEMINI_MODELS:
            provider_review = "gemini"

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial review
                if self.simulate:
                    raw_output = {
                        "model": model,
                        "temperature": temperature,
                        "reasoning_effort": reasoning_effort,
                        "chain_of_thought": chain_of_thought,
                        "simulated_review": (
                            f"This is a simulated review: Model='{model}', Temp={temperature}, "
                            f"Effort='{reasoning_effort}', chain_of_thought='{chain_of_thought}'. "
                        ),
                        "final_response": "PLACEHOLDER REVIEW"
                    }
                    structured_output = json.dumps(raw_output, indent=2)
                    return notes_content_for_df, structured_output, raw_output['final_response']

                # Call API
                else:
                    # For OpenAI
                    if provider_review == "openai":

                        # For reasoning models
                        if model in REASONING_MODELS:
                            kwargs = {}
                            if reasoning_effort:
                                kwargs["reasoning"] = {"effort": reasoning_effort}
                            if temperature is not None:
                                kwargs["temperature"] = float(temperature)

                            contract = "Return exactly ONE JSON object"
                            system_msg = {"role": "system", "content": "You are a helpful assistant."}
                            msgs = [system_msg, {"role": "user", "content": final_review_prompt + contract}]
                            resp = self._openai.chat.completions.create(
                                model=model,
                                messages=msgs,
                                max_tokens=max_tokens,
                                **kwargs,
                            )
                            raw_output = resp.choices[0].message.content or ""

                        # For non-reasoning models
                        else:
                            tools = [_schema_to_openai(chain_of_thought)]
                            ckwargs = {}
                            if temperature is not None:
                                ckwargs["temperature"] = float(temperature)

                            system_msg = {
                                "role": "system",
                                "content": "You are a parser. Call the function with exactly one JSON object that includes a text."
                            }
                            msgs = [system_msg, {"role": "user", "content": final_review_prompt}]
                            resp = self._openai.chat.completions.create(
                                model=model,
                                messages=msgs,
                                tools=tools if tools else None,
                                max_tokens=max_tokens,
                                **ckwargs,
                            )
                            choice = resp.choices[0]
                            if choice.message.tool_calls:
                                args = choice.message.tool_calls[0].function.arguments
                                raw_output = args
                            else:
                                raw_output = choice.message.content or ""

                    # For Groq
                    elif provider_review == "groq":
                        contract = "Return exactly ONE JSON object"
                        prompt_groq = final_review_prompt + contract
                        gkwargs = {}
                        if temperature is not None:
                            gkwargs["temperature"] = float(temperature)
                        resp = self._groq.chat.completions.create(
                            model=model,
                            messages=[{"role": "user", "content": prompt_groq}],
                            max_tokens=max_tokens,
                            **gkwargs,
                        )
                        raw_output = resp.choices[0].message.content.strip()

                    # For Gemini
                    elif provider_review == "gemini":
                        contract = "Return exactly ONE JSON object with a single key 'final_response'."
                        gemini_prompt = final_review_prompt + contract
                        gen_content_config_args = {
                            "response_mime_type": "application/json"
                        }
                        if model in REASONING_MODELS and reasoning_effort:
                            gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                        resp = self._gemini_client.models.generate_content(
                            model=model,
                            contents=gemini_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        raw_output = resp.text

                    parsed_output = _extract_json(raw_output)
                    return notes_content_for_df, raw_output, parsed_output.get('final_response', 'ERROR: Final response key missing from JSON.')

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Review Generation] provider={{provider_review}} model={{model}} attempt={{attempt}} -> {{e}}", flush=True)
                if attempt == max_retries:
                    # Case: Review generation fails. Return with successful notes and failure placeholders for review.
                    return notes_content_for_df, REVIEW_GENERATION_FAILURE, REVIEW_GENERATION_FAILURE
                time.sleep(retry_delay)

        # This point should theoretically not be reached if all success and failure paths return explicitly.
        # If it is reached, it implies an unhandled scenario, so we return a general failure for safety.
        return notes_content_for_df, UNKNOWN_ERROR_STATE, UNKNOWN_ERROR_STATE

### Helper for output messages

In [ ]:
def _fmt_combo(combo: ParamCombo) -> str:
    """Helper to format the ParamCombo into a human-readable string"""
    parts = []
    if combo.model is not None:
        parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None:
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    # Chain_of_thought can be an empty string
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    parts.append(f"note_taking={combo.note_taking}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    """
    Another helper to enrich the ParamCombo string with information for output messages"
    """
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

### Simulation definition

In [ ]:
def simulate_parameter_combo(
    combo: ParamCombo,
    review_target: Dict,
    client: Optional[LLMClient] = None,
) -> pd.DataFrame:

    """
    Iterates through the papers, creates the final user prompt,
    takes the configurational setting,
    calls the API/simulation NUM_REVIEWS_PER_PAPER times
    and collects the resulting reviews in a pandas DataFrame
    """

    # If client specified then use client, otherwise fake client for test
    client = client or LLMClient(simulate=True)

    # Use the full paper text as content input
    paper_content_for_llm = review_target['parsed_text']

    doc_id = review_target["paper_id"]

    # Initialize a dictionary for the current paper's results, including combo parameters
    current_paper_record = {
        "paper_id": doc_id,
        "model": combo.model,
        "temperature": combo.temperature,
        "reasoning_effort": combo.reasoning_effort,
        "chain_of_thought": combo.chain_of_thought,
        "note_taking": combo.note_taking
    }

    # Loop to generate multiple reviews (or notes) for each paper and parameter combo
    for review_idx in range(NUM_REVIEWS_PER_PAPER):

        log_call(doc_id, combo, review_number=review_idx + 1)

        notes_content, raw, parsed_review_string = client.call( # Capture parsed as string directly
            model=combo.model,
            initial_prompt_content=paper_content_for_llm,
            temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
            reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
            chain_of_thought=combo.chain_of_thought,
            note_taking_active=combo.note_taking
        )

        # Store notes (this line is commented out to exclude notes from the dataframe)
        current_paper_record[f"llm_notes_{review_idx + 1}"] = notes_content
        # Store raw review
        current_paper_record[f"llm_review_{review_idx + 1}"] = raw
        # Store parsed review (now directly a string)
        current_paper_record[f"parsed_llm_review_{review_idx + 1}"] = parsed_review_string

    # Convert the single record into a DataFrame, explicitly passing an index
    df = pd.DataFrame(current_paper_record, index=[0])
    return df

### Configuration grid

In [ ]:
# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS + GEMINI_MODELS

# Generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, NOTE_TAKING))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")

display(experiment_config)

### Helper for logging

In [ ]:
def _nan_to_none(x):

    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x


def combo_key_tuple(row) -> tuple:

    """Combines the congfigurational setting with None instead of NaN"""

    return (
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["note_taking"]
    )

def combo_key_str(row) -> str:

    """Creates readable combo string for logging"""

    t = combo_key_tuple(row)
    return f"model={t[0]}|temp={t[1]}|re={t[2]}|cot={t[3]}|notes={t[4]}"

### LLM review generation

In [ ]:
if __name__ == '__main__':

    # Print information
    print(f"Processing and saving results to: {RESULTS_PATH}", flush = True)
    print(f"Logging successful iterations to: {LOG_PATH}", flush = True)

    # Initialize client
    client = LLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    # Initialize keys as a set of (paper_id, combo_str) tuples
    done_keys = set()

    # Check and read successful paper-parameter combos from log file
    if os.path.exists(LOG_PATH):
        try:
            log_df = pd.read_parquet(LOG_PATH)
            # Corrected: Use 'completed_combo' consistently
            if 'paper_id' in log_df.columns and 'completed_combo' in log_df.columns:
                done_keys.update(zip(log_df['paper_id'].tolist(), log_df['completed_combo'].tolist()))
                print(f"Loaded {len(done_keys)} completed paper-combo entries from log.")
            else:
                print(f"Warning: '{LOG_PATH}' exists but does not contain 'paper_id' and 'completed_combo' columns. Starting with empty log.")
        except Exception as e:
            print(f"Error reading existing log parquet file: {e}. Starting with empty log.")
            done_keys = set()

    # Check and read successful paper-parameter combos from results file (for robustness and consistency)
    if os.path.exists(RESULTS_PATH):
        existing_df = pd.read_parquet(RESULTS_PATH)
        if not existing_df.empty:
            # Define all columns that define a unique paper-parameter combo
            needed_col_for_grouping = ["paper_id", "model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

            ex = existing_df.copy()

            # Ensure no nessecary column is missing (they should not)
            missing_cols = [c for c in needed_col_for_grouping if c not in ex.columns]

            if not missing_cols:
                # Normalize NAs to None for keys (only for relevant columns)
                for c in ["temperature", "reasoning_effort"]:
                    ex[c] = ex[c].where(pd.notna(ex[c]), None)

                # Count amount of reviews per unqiue combo
                paper_combo_counts = (
                    ex.groupby(needed_col_for_grouping)
                    .size()
                    .reset_index(name='review_count')
                )

                # Identify paper-configuration combos that have the expected number of reviews
                for _, r in paper_combo_counts.iterrows():
                    if int(r["review_count"]) == NUM_REVIEWS_PER_PAPER:
                        # Load `paper_id` and `completed_configuration_str` into the keys
                        completed_configuration_str = combo_key_str(r)
                        done_keys.add((r['paper_id'], completed_configuration_str))

    print(f"Already-completed paper-configuration combos: {len(done_keys)}")

    # Start iteration
    # Outer loop: iterate over parameter combinations
    for idx_combo, row_combo in tqdm(experiment_config.iterrows(), total=len(experiment_config), desc="Processing Combos"):

        combo_str_current = combo_key_str(row_combo) # String representation of the current combo

        # Inner loop: iterate over each paper
        for idx_paper, paper_data in enumerate(targets):
            paper_id = paper_data["paper_id"]
            paper_combo_key = (paper_id, combo_str_current)

            # Check if this specific paper-combo is already completed
            if paper_combo_key in done_keys:
                print(f"[SKIP] Paper {paper_id} with combo {combo_str_current} already complete.")
                continue

            # If not, construct current combo (ParamCombo object)
            combo = ParamCombo(
                model=row_combo["model"],
                temperature=None if pd.isna(row_combo["temperature"]) else float(row_combo["temperature"]),
                reasoning_effort=None if pd.isna(row_combo["reasoning_effort"]) else str(row_combo["reasoning_effort"]),
                chain_of_thought=row_combo["chain_of_thought"],
                note_taking=row_combo["note_taking"]
            )

            print(f"\n[RUN] Combo {idx_combo+1}/{len(experiment_config)}, Paper {idx_paper+1}/{len(targets)} -> {paper_combo_key[0]} | {paper_combo_key[1]}", flush=True)

            try:
                # Make actual simulation / call for this single paper
                df_combo_paper = simulate_parameter_combo(combo, paper_data, client=client)

                # Append results to RESULTS_PATH
                if os.path.exists(RESULTS_PATH):
                    # Read, concat and save
                    existing_df = pd.read_parquet(RESULTS_PATH)
                    combined_df = pd.concat([existing_df, df_combo_paper], ignore_index=True)
                    combined_df.to_parquet(RESULTS_PATH, index=False)
                else:
                    # Write a new result file
                    df_combo_paper.to_parquet(RESULTS_PATH, index=False)

                # Add done keys as soon as result is stored (liberal)
                done_keys.add(paper_combo_key)

                # Log successful paper-parameter combo (i.e., NUM_REVIEWS_PER_PAPER have been generated for this paper-parameter combo)
                new_log_entry_df = pd.DataFrame(
                    {
                        'paper_id': paper_id,
                        'completed_combo': combo_str_current
                    },
                    index=[0]
                )

                try:
                    if os.path.exists(LOG_PATH):
                        # Read, concat, drop duplicates and save
                        existing_log_df = pd.read_parquet(LOG_PATH)
                        combined_log_df = pd.concat([existing_log_df, new_log_entry_df], ignore_index=True)
                        combined_log_df.drop_duplicates(subset=['paper_id', 'completed_combo'], inplace=True)
                        combined_log_df.to_parquet(LOG_PATH, index=False)
                    else:
                        # Write a new log file
                        new_log_entry_df.to_parquet(LOG_PATH, index=False)

                except Exception as log_e:
                    print(f"Warning: Could not append to existing log parquet file {LOG_PATH}: {log_e}. Task (paper: {paper_id}, combo: {combo_str_current}) is still marked as done based on results file.", flush=True)

            # Catch a code error (before the result is stored and the key is added)
            except Exception as e:
                print(f"[ERROR] Paper {paper_id} with combo {combo_str_current} -> {type(e).__name__}: {e}", flush=True)

In [72]:
# result.parquet-file
llm_results_df = pd.read_parquet(RESULTS_PATH)

# Check results
display(llm_results_df.head(5))
display(llm_results_df.shape)

# log.parquet-file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check if the expected column exists, otherwise display full dataframe
        if 'completed_combo' in llm_log_df.columns:
            display(llm_log_df[['completed_combo']].head(5))
            display(llm_log_df.shape)
        else:
            print(f"Warning: Log file '{LOG_PATH}' exists but does not contain 'completed_combo' column.")
            display(llm_log_df.head(5))
            display(llm_log_df.shape)
    except Exception as e:
        print(f"Error reading log parquet file for display: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,scope,note_taking,llm_notes_1,llm_review_1,parsed_llm_review_1
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,abstract,Yes,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,abstract,Yes,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
2,XZRmNjUMj0c,gpt-5-mini-2025-08-07,NaN,low,,abstract,Yes,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
3,cDYRS5iZ16f,gpt-5-mini-2025-08-07,NaN,low,,abstract,Yes,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
4,Sh97TNO5YY_,gpt-5-mini-2025-08-07,NaN,low,,abstract,Yes,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW


(20384, 10)

,completed_combo
0,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...
1,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...
2,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...
3,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...
4,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...


(20384, 2)

### References

The main code logic and in parts exact code has been taken from:

Cummins, J. (2025). The threat of analytic flexibility in using large language models to simulate human data: A call to attention. *arXiv preprint* arXiv:2509.13397.